# 02 — Y2Y corridor-wide solve (Gate-0 arms)

Runs prioritizr on the **full** aligned stack (no crop) — the corridor-wide solve, over the shared
`prioritizr_core.R` engine. Parameters come from `config.ANALYSES["y2y"]`, carried via
`manifest.json` (cell 1 refreshes it).

**Configure a run: edit cell 2, the RUN LEVER.** It carries the **Gate-0 arms** of the
frequency-ensemble study (`analyses/y2y/spec/`): `a0_control` / `a1_protocol` / `a2_flat30` /
`a3_flat40` (+ optional `a4_pullcheck`), each a `(TARGETS, WEIGHTS)` pair with **w = t** so only
the stopping point varies between arms. Uncomment one block, run top-to-bottom, repeat — **or just Run All once**: the final BATCH cell solves every remaining arm unattended (resumable; solved arms are skipped). The arm
dicts come from the frozen Gate-0a table — run `01_feature_audit.ipynb` first;
verdicts come after — run `03_gate0_validation.ipynb` when all arms are solved.

Overrides are applied to the run context, not written back into `manifest.json`;
`run_summary.json` records the parameters **actually solved**, so that is the file to compare
runs on. `pr_override` refuses to overwrite a folder whose recorded targets *or weights* differ.

**Solver mode (2026-08-26, Gurobi live):** the lever's `MODE` switch selects the **production formulation** — binary decisions + Gurobi at opt_gap 0.01% with numeric_focus, folders `iter8_y2y_<arm>` — by default; the earlier proportion-LP arms (`iter7_…`, HiGHS) stay on disk as the relaxation-tightness record. Gurobi's WLS licence needs **live internet during solves**.

**Kernel:** `R (y2y)`. Run cell-by-cell; Ethan runs, Claude never executes.

In [1]:
# ---- Setup: shared engine + this analysis' key + manifest refresh --------
# (source() moved below the bootstrap so the path resolves from any cwd)
ANALYSIS <- "y2y"            # <-- 03b/03c (still at repo root) differ only in this line
# Root-finding bootstrap: this notebook lives in analyses/y2y/, below the repo root where
# config.py and prioritizr_core.R sit. Same pattern as run_one.R.
PROJ <- normalizePath(getwd())
while (!file.exists(file.path(PROJ, "config.py"))) {
  parent <- dirname(PROJ)
  if (identical(parent, PROJ)) stop("config.py not found above getwd() -- open from inside the repo")
  PROJ <- parent
}
setwd(PROJ)
source(file.path(PROJ, "prioritizr_core.R"))   # pr_* functions

mpath <- pr_refresh_manifest(PROJ, ANALYSIS)   # regenerate manifest.json from config for THIS
ctx   <- pr_setup(mpath, PROJ)                 # analysis (stops on failure); print the banner

manifest refreshed from config.py (analysis=y2y)
prioritizr 8.1.0 | terra 1.9.34 | analysis=y2y | solver=highs (single solution)
objective=min_shortfall | budget=30% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=full | lock_in=pa_mask
penalties: connectivity=0 | boundary=0 | neighbor=0
outputs -> output_data/iter6_y2y


In [2]:
# ============ THE RUN LEVER — Gate-0 arms: edit THIS cell, run the notebook ============
# One arm per pass: uncomment exactly ONE block, run top-to-bottom, repeat -- or ignore
# this lever entirely and Run All: the BATCH cell at the end solves all remaining arms. Everything else comes
# from config.py. The arm dicts are DERIVED by 01_feature_audit.ipynb (this folder) from the frozen
# Gate-0a table -- run that notebook first; its last cell prints these blocks, and this cell must
# agree with it. Verdicts afterwards: 03_gate0_validation.ipynb (this folder).
#
# A target is a STOPPING RULE. Under min-shortfall the objective depends on w/t below target
# ("pull") and is flat above it. ALL ARMS SET WEIGHTS <- TARGETS (w = t), so pull stays 1.00 for
# every feature and the ONLY thing varying across arms is the stopping point -- at w = 1 a target
# of 0.332 raises carbon's pull to 3.0x and its share of objective swing to ~55%, confounding the
# comparison (that superseded configuration ran once as iter7_y2y_r1_density5x).
#
# MEASURED, from that run (2026-08-18): (1) m_soc parked at EXACTLY its 0.332 target; (2) biomass
# landed at 0.259 vs its 0.066 target -- min-shortfall never penalizes EXCEEDING a target, so a
# satiated feature free-rides on cells picked for other values; excess above target is expected
# co-capture, not a failure. (3) solve time 4,289 s (~71 min, almost all HiGHS presolve) -- NOT
# the ~12 s of the untargeted iter2 LP. Expect a0 fast and a1-a3 possibly ~1 h each.
#
# Do NOT add targets to the foundational features: if every target became simultaneously
# achievable the objective would go flat and the solver would return an arbitrary optimum.

# ---- the Gate-0 arms — uncomment exactly ONE ----------------------------------
# a0 control: no carbon change. Isolates penalty-removal + 1/v; brackets S4 (extreme carbon-forward).
RUN <- "a0_control";  TARGETS <- list()

# a1 the protocol configuration -- m_soc ONLY (biomass REVERTED to weight-levered by R2's
#    tail-mass criterion: implied target 0.066 < t_min 0.15; its ~40%+ capture here is expected).
# RUN <- "a1_protocol"; TARGETS <- list(irrecoverable_carbon_m_soc = 0.332)

# a2 flat 30%: both pools at exactly area share.
# RUN <- "a2_flat30";   TARGETS <- list(irrecoverable_carbon_m_soc   = 0.30,
#                                       irrecoverable_carbon_biomass = 0.30)

# a3 flat 40%: mild demotion; biomass binds barely (control captures 41.8%) -- watch it.
# RUN <- "a3_flat40";   TARGETS <- list(irrecoverable_carbon_m_soc   = 0.40,
#                                       irrecoverable_carbon_biomass = 0.40)

# a4 OPTIONAL pull-invariance check (gate G-uniform). t must be UNREACHABLE so the target can do
#    nothing (connectivity cap_max = 0.552 -> t = 0.6 is beyond any feasible capture); then
#    w = t = 0.6 leaves pull at 1.00 and the solution MUST reproduce a0 exactly -- the empirical
#    proof that only w/t and the stopping point matter.
# RUN <- "a4_pullcheck"; TARGETS <- list(transboundary_connectivity = 0.6)

# a4 DIAGNOSTIC re-solve (2026-08-26). The first binary a4 returned an objective 0.42%
#    above its KNOWN optimum (5.158146 exactly -- the integral LP relaxation proves it),
#    which is inconsistent with a clean 0.1%-gap stop; its Gurobi log was lost unsaved.
#    This arm re-solves at gap 1e-4 into a NEW folder. Expected: objective <= 5.1587 and
#    the anomaly attributed to the first solve's termination; if it AGAIN parks ~5.18,
#    something structural is wrong in the a4 model -- stop and report. SAVE the notebook
#    after the run so the Gurobi summary lines survive this time.
# RUN <- "a4_pullcheck_v2"; TARGETS <- list(transboundary_connectivity = 0.6); OPT_GAP_ARM <- 1e-4

# a4 v3 -- FIX VALIDATION (2026-08-26), ACTIVE ARM. Root cause of the v1/v2 anomaly found:
#    with the matrix spanning [1e-11, 1e5], Gurobi WITHOUT NumericFocus mis-converged its
#    root LP 0.42% high and certified a FALSE optimum (v2 reproduced it bit-identically --
#    deterministic numerics, not a loose stop). The engine now passes numeric_focus=TRUE
#    (Gurobi NumericFocus 2) on every solve. EXPECT: objective <= ~5.1582 (true optimum
#    5.158146, known exactly from the integral LP twin). May run slower than the 18-min v2.
#    SAVE the notebook after the run so the Gurobi log survives.
# RUN <- "a4_pullcheck_v3"; TARGETS <- list(transboundary_connectivity = 0.6); OPT_GAP_ARM <- 1e-4  # DONE: 17 s, exact
# ------------------------------------------------------------------------------

# ---- SOLVER MODE (added 2026-08-26, Gurobi licence live: 200k-var probe OPTIMAL in 12.6 s) ----
# "milp_gurobi" = the PRODUCTION formulation: binary decisions + Gurobi, opt_gap tightened to
#     0.1% so "capture binds at target" is a claim about a near-proven optimum, portfolio off
#     (single solution). Folders: iter8_y2y_<arm>. Needs LIVE INTERNET (WLS licence checks out
#     over the network during optimization).
# "lp_highs"    = the proportion-LP relaxation on HiGHS -- the iter7 generation, KEPT on disk as
#     the measured relaxation-tightness record. Re-select only to reproduce those.
MODE    <- "milp_gurobi"
# An arm may set OPT_GAP_ARM above to tighten its own solve (used once, then cleared, so
# switching arms cannot inherit a stale gap).
# STANDARD = 1e-4 (adopted 2026-08-26 per the pre-committed rule: the a4_v3 validation
# solve hit the exact optimum in 17 s under numeric_focus -- tight certificates are cheap
# here because the LP relaxation is near-integral, so the root bound is already tight).
OPT_GAP <- if (exists("OPT_GAP_ARM")) OPT_GAP_ARM else 1e-4
if (exists("OPT_GAP_ARM")) rm(OPT_GAP_ARM)
OV      <- if (MODE == "milp_gurobi") list(solver = "gurobi", decision_type = "binary",
                                           opt_gap = OPT_GAP, portfolio_n = 1) else list()
PREFIX <- if (MODE == "milp_gurobi") "iter8_y2y_" else "iter7_y2y_"

WEIGHTS <- TARGETS    # w = t on every arm: pull 1.00; only the stopping point varies

# DIRECT assignment, deliberately NOT `modifyList(ctx, pr_override(...))`: modifyList deep-merges
# nested lists, so an arm's TARGETS would MERGE with config.py's baseline instead of replacing it
# (a0's empty list would clear nothing and the control would quietly solve the baseline target).
# pr_override returns the full updated ctx and prints the EFFECTIVE targets/weights -- on a0 that
# line must read "<none>"; check it before solving.
ctx <- do.call(pr_override, c(list(ctx,
    targets                    = TARGETS,
    feature_weight_multipliers = WEIGHTS,
    results_subdir             = paste0(PREFIX, RUN)), OV))

  override targets          -> {} (empty -- cleared to defaults)
  override feature_weight_multipliers -> {} (empty -- cleared to defaults)
  override results_subdir   -> iter8_y2y_a0_control
  override solver           -> gurobi
  override decision_type    -> binary
  override opt_gap          -> 1e-04
  override portfolio_n      -> 1
  EFFECTIVE targets: <none> | weight multipliers: <none>
  outputs  -> output_data/iter8_y2y_a0_control


In [3]:
# ---- Ingest the stack + crop to the ROI + normalize (window set in config.ANALYSES) ----
ctx <- modifyList(ctx, pr_ingest(ctx))

ingested 48 features (8 continuous + 40 EFG) + cost + PA mask | grid 1286 x 3312 @ 1000 m
normalized 48 features to total=100000 each (scale-invariant conditioning)


In [4]:
# ---- Planning units + lock-in + feasibility check ----
ctx <- modifyList(ctx, pr_planning_units(ctx))

planning units: 1,272,914 cells | budget = 30% = 381,874 cells
locked-in [pa_mask]: 191,029 cells (15.0% of window) -- fits within budget


In [5]:
# ---- Feature weights (+ any per-analysis up-weighting) ----
ctx <- modifyList(ctx, pr_weights(ctx))

weights: 8 continuous @ 1.0 ; 40 EFG @ 0.0250 (EFG group total = 1.0)


In [6]:
# ---- Per-feature relative targets (from the RUN LEVER cell above) ----
# Applies the target vector: every feature at the default target_pct, then the overrides. Built
# BY NAME off names(features), so it cannot silently misalign if the stack order changes; an
# unknown feature name or an out-of-range value stops the run here rather than at the solve.
# Check the printed overrides match the lever cell before solving.
ctx <- modifyList(ctx, pr_targets(ctx))

targets: 1.00 for all 48 features (no overrides)


In [7]:
# ---- Spatial-penalty matrices (built only for penalties > 0) ----
ctx <- modifyList(ctx, pr_penalty_matrices(ctx))

penalties -> connectivity=0 | boundary=0 | neighbor=0  (0 = off)


In [8]:
# ---- Build the conservation problem ----
bp <- pr_build_problem(ctx); ctx$p <- bp$p; ctx$solve_params <- bp$solve_params

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (48 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (all equal to 1)
││└•weights:    continuous values (between 0.025 and 1)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



In [9]:
# ---- Solve (Ethan runs; heavy) -- HiGHS single solution / Gurobi portfolio ----
# A run that hits the time limit returns an INFEASIBLE point (area > budget) -- discard it.
sv <- pr_solve(ctx); ctx$s <- sv$s; ctx$timing <- sv$timing; ctx$n_sol <- sv$n_sol

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.5.0 25F84)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 49 rows, 1272962 columns and 16912559 nonzeros (Min)
Model fingerprint: 0xe852112c
Model has 48 linear objective coefficients
Variable types: 48 continuous, 1272914 integer (1272914 binary)
Coefficient statistics:
  Matrix range     [1e-04, 1e+05]
  Objective range  [3e-02, 1e+00]
  Bounds range     [1e+00, 1e+0

In [10]:
# ---- Per-alternative summaries + selection-frequency map ----
ctx <- modifyList(ctx, pr_summaries(ctx))

  alternative n_selected pct_region n_added_beyond_pa
1      alt_01     381874   29.99998            190845


In [11]:
# ---- Write outputs for 04 (portfolio / frequency / representation / run_summary) ----
pr_write_outputs(ctx)

wrote:
  output_data/iter8_y2y_a0_control/portfolio.tif
  output_data/iter8_y2y_a0_control/selection_frequency.tif
  output_data/iter8_y2y_a0_control/portfolio_representation.csv
  output_data/iter8_y2y_a0_control/run_summary.json


## Batch mode — every remaining arm in one pass

The alternative to editing the lever four times: this cell loops the Gate-0 arms through the same
engine stages, **skipping any arm already solved** (its `run_summary.json` exists), so it is
resumable after an interruption and idempotent on re-run.

**One-click campaign: just Run All.** The single-arm cells above solve the lever's default arm
(`a0_control`, fast); this cell then solves the rest (a1–a3 possibly ~1 h each, a4 fast). Each
arm still writes its own `iter7_y2y_<arm>` folder, prints its `EFFECTIVE targets:` tripwire, and
passes the clobber guard — identical records to the manual path, just unattended.

Per-arm state is a fresh copy of the base context (`pr_override` returns a modified copy), so arms
cannot leak settings into each other; features and normalization are ingested once and reused.

In [12]:
# ---- BATCH: solve every remaining Gate-0 arm (resumable; skips solved arms) ----
ARMS <- list(
  a0_control   = list(),
  a1_protocol  = list(irrecoverable_carbon_m_soc = 0.332),   # from the FROZEN Gate-0a table
  a2_flat30    = list(irrecoverable_carbon_m_soc   = 0.30,
                      irrecoverable_carbon_biomass = 0.30),
  a3_flat40    = list(irrecoverable_carbon_m_soc   = 0.40,
                      irrecoverable_carbon_biomass = 0.40),
  a4_pullcheck = list(transboundary_connectivity = 0.6)      # unreachable (cap_max 0.552); under the
                      # LP this reproduced a0 EXACTLY (0 cells differ); under binary MILP expect the
                      # SAME OBJECTIVE but possibly a different optimum among near-ties -- that
                      # divergence is itself a degeneracy datum for Gate 2
)

# MODE/OV/PREFIX come from the lever cell; default to the production formulation if the
# lever was skipped (e.g. running only this cell after cells 1, 3, 4).
if (!exists("PREFIX")) { MODE <- "milp_gurobi"
  OV <- list(solver = "gurobi", decision_type = "binary", opt_gap = 0.001, portfolio_n = 1)
  PREFIX <- "iter8_y2y_" }
cat(sprintf("batch mode: %s -> %s<arm>\n", MODE, PREFIX))
t_batch <- proc.time()[["elapsed"]]
for (arm in names(ARMS)) {
  subdir <- paste0(PREFIX, arm)
  if (file.exists(file.path(PROJ, "output_data", subdir, "run_summary.json"))) {
    cat(sprintf("== %-13s already solved -- skipped\n", arm)); next
  }
  cat(sprintf("\n===================== %s =====================\n", arm))
  # fresh per-arm context: pr_override returns a modified COPY of ctx, and w = t on every arm
  actx <- do.call(pr_override, c(list(ctx, targets = ARMS[[arm]],
                      feature_weight_multipliers = ARMS[[arm]],
                      results_subdir = subdir), OV))
  actx <- modifyList(actx, pr_weights(actx))
  actx <- modifyList(actx, pr_targets(actx))
  actx <- modifyList(actx, pr_penalty_matrices(actx))
  bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
  sv <- pr_solve(actx); actx$s <- sv$s; actx$timing <- sv$timing; actx$n_sol <- sv$n_sol
  actx <- modifyList(actx, pr_summaries(actx))
  pr_write_outputs(actx)
  cat(sprintf("== %s done in %.0f s | batch elapsed %.1f min\n",
              arm, sv$timing[["elapsed"]], (proc.time()[["elapsed"]] - t_batch) / 60))
}
cat("\nBATCH COMPLETE -- next: analyses/y2y/03_gate0_validation.ipynb\n")

batch mode: milp_gurobi -> iter8_y2y_<arm>
== a0_control    already solved -- skipped

===================== a1_protocol =====================
  override targets          -> irrecoverable_carbon_m_soc=0.332
  override feature_weight_multipliers -> irrecoverable_carbon_m_soc=0.332
  override results_subdir   -> iter8_y2y_a1_protocol
  override solver           -> gurobi
  override decision_type    -> binary
  override opt_gap          -> 1e-04
  override portfolio_n      -> 1
  EFFECTIVE targets: irrecoverable_carbon_m_soc=0.332 | weight multipliers: irrecoverable_carbon_m_soc=0.332
  outputs  -> output_data/iter8_y2y_a1_protocol
weights: 8 continuous @ 1.0 ; 40 EFG @ 0.0250 (EFG group total = 1.0)
  up-weight irrecoverable_carbon_m_soc x0.3 -> 0.3320
targets: default 1.00; 1 override(s):
  irrecoverable_carbon_m_soc         0.332
penalties -> connectivity=0 | boundary=0 | neighbor=0  (0 = off)


A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (48 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.332 and 1)
││└•weights:    continuous values (between 0.025 and 1)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.5.0 25F84)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 49 rows, 1272962 columns and 16912559 nonzeros (Min)
Model fingerprint: 0xfe19235a
Model has 48 linear objective coefficients
Variable types: 48 continuous, 1272914 integer (1272914 binary)
Coefficient statistics:
  Matrix range     [1e-04, 1e+05]
  Objective range  [3e-02, 1e+00]
  Bounds range     [1e+00, 1e+0

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (48 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.3 and 1)
││└•weights:    continuous values (between 0.025 and 1)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.5.0 25F84)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 49 rows, 1272962 columns and 16912559 nonzeros (Min)
Model fingerprint: 0xe6d0e5c1
Model has 48 linear objective coefficients
Variable types: 48 continuous, 1272914 integer (1272914 binary)
Coefficient statistics:
  Matrix range     [1e-04, 1e+05]
  Objective range  [3e-02, 1e+00]
  Bounds range     [1e+00, 1e+0

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (48 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.4 and 1)
││└•weights:    continuous values (between 0.025 and 1)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.5.0 25F84)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 49 rows, 1272962 columns and 16912559 nonzeros (Min)
Model fingerprint: 0x5c72bfcc
Model has 48 linear objective coefficients
Variable types: 48 continuous, 1272914 integer (1272914 binary)
Coefficient statistics:
  Matrix range     [1e-04, 1e+05]
  Objective range  [3e-02, 1e+00]
  Bounds range     [1e+00, 1e+0

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (48 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.6 and 1)
││└•weights:    continuous values (between 0.025 and 1)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.5.0 25F84)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 49 rows, 1272962 columns and 16912559 nonzeros (Min)
Model fingerprint: 0x8429897d
Model has 48 linear objective coefficients
Variable types: 48 continuous, 1272914 integer (1272914 binary)
Coefficient statistics:
  Matrix range     [1e-04, 1e+05]
  Objective range  [3e-02, 1e+00]
  Bounds range     [1e+00, 1e+0